In [62]:
import numpy as np
from scipy.linalg import expm

phi1 = 1.3758
phi2 = -0.3921

phi=np.array([phi1,phi2])
print("arma coefficients :", phi)

if phi2>0:
    print("problem: phi2 must be <0")

coeffarma = [-phi2, -phi1, 1]

roots = np.roots(coeffarma)
print("roots :", roots)

lambda_vals = -np.log(roots) 
print("lambda :", lambda_vals)

def coeffcarma(eigenvalues):
    [lam1,lam2] = eigenvalues.copy()
    a1=-(lam1+lam2)
    a2=lam1*lam2
    return [a1,a2]

a=coeffcarma(lambda_vals)
a= np.array([float(x) for x in a])

if phi1-2*np.exp(-a[0]/2)<0:
    print("problem: phi1 must be >= 2*exp(-a1)/2")

print("carma AR coefficients :", a)

theta=-0.1304

print("arma MA coefficients :", theta)

arma coefficients : [ 1.3758 -0.3921]
roots : [2.48072398 1.0280748 ]
lambda : [-0.90855044 -0.02768793]
carma AR coefficients : [0.93623837 0.02515588]
arma MA coefficients : -0.1304


In [ ]:
import numpy as np

def buildcompanion(a):
    p = len(a)
    A = np.zeros((p,p))
    for i in range(p-1):
        A[i,i+1] = 1

    A[p-1,:] = -np.array(a[::-1])
    return A

buildcompanion(a)

array([[ 0.        ,  1.        ],
       [-0.02515588, -0.93623837]])

In [ ]:
import numpy as np
from scipy.linalg import solve_continuous_lyapunov

def buildsigma(coefcarma):
    A=buildcompanion(coefcarma)
    p=len(coefcarma)
    ep=np.zeros((p,1))
    ep[p-1,0]=1

    Q=ep@ep.T

    sigma=solve_continuous_lyapunov(A,-Q)
    return sigma
buildsigma(a)

array([[2.12297119e+01, 1.66533454e-15],
       [1.67053871e-15, 5.34052028e-01]])

In [ ]:
def m0(phi,a):
    A=buildcompanion(a)
    [phi1,phi2]=phi
    sigma=buildsigma(a)
    M0=((1+phi1**2+phi2**2)*np.identity(2)+(2*phi1*phi2-2*phi1)*expm(A)-2*phi2*expm(2*A))@sigma
    return M0

def m1(phi,a):
    A=buildcompanion(a)
    [phi1,phi2]=phi
    sigma=buildsigma(a)
    M1=((-phi1+phi1*phi2)*np.identity(2)+(1-phi2+phi1**2+phi2**2)*expm(A)+(-phi1+phi1*phi2)*expm(2*A)-phi2*expm(3*A))@sigma
    return M1

def m(phi,a,theta):
    M0=m0(phi,a)
    M1=m1(phi,a)
    return M0-((1+theta**2)/theta)*M1

In [ ]:
M=m(phi,a,theta)

b0=(1/(2*M[0,0]))*(-(M[0,1]+M[1,0])+np.sqrt((M[0,1]+M[1,0])**2-4*M[0,0]*M[1,1]))

print("carma MA coefficient :", b0)
print("carma AR coefficients :", a)


carma MA coefficient : 1.786374352354718
carma AR coefficients : [0.93623837 0.02515588]
